<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS06/Praktikum6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 17.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 20.5 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520959 sha256=a3f2e7d840ecb72511dabf5773c27ff2ffce4fde0c56fddb7429d6136c452201
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679067 sha256=cf17a203dd32f25d7e4e6ee29577e0060d66b4f0e4aa2449e48fea4df3548f86
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built annoy hnswlib


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import time
import faiss

from annoy import AnnoyIndex
import hnswlib

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Load dataset
file_path = '/content/drive/MyDrive/songs_with_attributes_and_lyrics.csv'
df = pd.read_csv(file_path)

# Fitur numerik sesuai modul
features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo'
]

# Periksa apakah kolom tersedia
missing = [col for col in features if col not in df.columns]

if missing:
    raise ValueError(f"Kolom tidak ditemukan di CSV: {missing}")

# Mengubah fitur menjadi numerik dan menangani nilai kosong
numeric_data = df[features].apply(pd.to_numeric, errors='coerce')
valid_rows = numeric_data.notna().all(axis=1)

df_clean = df.loc[valid_rows].reset_index(drop=True)
X = numeric_data.loc[valid_rows].to_numpy(dtype=np.float32)

print("Jumlah data awal:", len(df))
print("Jumlah data setelah pembersihan:", len(X))
print("Jumlah fitur:", X.shape[1])

Jumlah data awal: 955320
Jumlah data setelah pembersihan: 955320
Jumlah fitur: 9


In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Query menggunakan lagu pertama
query_idx = 0
query = X_scaled[query_idx:query_idx + 1]

# Cari 5 tetangga selain lagu query itu sendiri
k = 5
search_k = k + 1

print("Normalisasi selesai.")
print("Ukuran dataset:", X_scaled.shape)

Normalisasi selesai.
Ukuran dataset: (955320, 9)


In [5]:
print("=== Exact NN ===")

nn = NearestNeighbors(
    n_neighbors=search_k,
    algorithm='brute',
    metric='euclidean'
)

start = time.time()
nn.fit(X_scaled)
exact_build_time = time.time() - start

start = time.time()
dist_exact, idx_exact = nn.kneighbors(query)
exact_query_time = time.time() - start

# Mengeluarkan lagu query dari hasil
exact_results = [
    (int(i), float(d))
    for i, d in zip(idx_exact[0], dist_exact[0])
    if i != query_idx
][:k]

print("Top-5 neighbors:", [i for i, d in exact_results])
print("Build time:", round(exact_build_time, 4), "detik")
print("Query time:", round(exact_query_time, 4), "detik")

=== Exact NN ===
Top-5 neighbors: [394553, 764272, 837727, 749223, 61511]
Build time: 0.0352 detik
Query time: 0.0869 detik


In [7]:
print("\n=== Annoy ===")

f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')

start = time.time()

for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)

index_annoy.build(10)
annoy_build_time = time.time() - start

start = time.time()
idx_annoy, dist_annoy = index_annoy.get_nns_by_vector(
    query[0], search_k, include_distances=True
)
annoy_query_time = time.time() - start

annoy_results = [
    (int(i), float(d))
    for i, d in zip(idx_annoy, dist_annoy)
    if i != query_idx
][:k]

print("Top-5 neighbors:", [i for i, d in annoy_results])
print("Build time:", round(annoy_build_time, 4), "detik")
print("Query time:", round(annoy_query_time, 4), "detik")


=== Annoy ===
Top-5 neighbors: [764272, 837727, 749223, 61511, 625227]
Build time: 9.8582 detik
Query time: 0.0003 detik


In [8]:
print("\n=== HNSW ===")

p_hnsw = hnswlib.Index(
    space='l2',
    dim=X_scaled.shape[1]
)

start = time.time()

p_hnsw.init_index(
    max_elements=len(X_scaled),
    ef_construction=200,
    M=16
)
p_hnsw.add_items(X_scaled)
hnsw_build_time = time.time() - start

p_hnsw.set_ef(200)

start = time.time()
idx_hnsw, dist_hnsw = p_hnsw.knn_query(
    query, k=search_k
)
hnsw_query_time = time.time() - start

hnsw_results = [
    (int(i), float(d))
    for i, d in zip(idx_hnsw[0], dist_hnsw[0])
    if i != query_idx
][:k]

print("Top-5 neighbors:", [i for i, d in hnsw_results])
print("Build time:", round(hnsw_build_time, 4), "detik")
print("Query time:", round(hnsw_query_time, 4), "detik")


=== HNSW ===
Top-5 neighbors: [394553, 764272, 837727, 749223, 61511]
Build time: 138.6128 detik
Query time: 0.0005 detik


In [13]:
print("\n=== FAISS IVF ===")

dim = X_scaled.shape[1]
nlist = 100

quantizer = faiss.IndexFlatL2(dim)

# METRIC_L2 diberikan sebagai argumen posisi
index_faiss = faiss.IndexIVFFlat(
    quantizer,
    dim,
    nlist,
    faiss.METRIC_L2
)

start = time.time()
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
faiss_build_time = time.time() - start

index_faiss.nprobe = 10

start = time.time()
dist_faiss, idx_faiss = index_faiss.search(
    query, search_k
)
faiss_query_time = time.time() - start

# Mengeluarkan lagu query dari hasil
faiss_results = [
    (int(i), float(d))
    for i, d in zip(idx_faiss[0], dist_faiss[0])
    if i >= 0 and i != query_idx
][:k]

print("Top-5 neighbors:", [i for i, d in faiss_results])
print("Build time:", round(faiss_build_time, 4), "detik")
print("Query time:", round(faiss_query_time, 4), "detik")



=== FAISS IVF ===
Top-5 neighbors: [394553, 764272, 837727, 749223, 61511]
Build time: 0.4933 detik
Query time: 0.0034 detik


In [14]:
# Mengumpulkan hasil
comparison = [
    ["Exact NN", exact_build_time, exact_query_time,
     [i for i, d in exact_results]],
    ["Annoy", annoy_build_time, annoy_query_time,
     [i for i, d in annoy_results]],
    ["HNSW", hnsw_build_time, hnsw_query_time,
     [i for i, d in hnsw_results]],
    ["FAISS IVF", faiss_build_time, faiss_query_time,
     [i for i, d in faiss_results]]
]

print("\n=== TABEL PERBANDINGAN ===")
print("-" * 100)
print(
    f"{'Metode':<15}"
    f"{'Build Time (s)':<20}"
    f"{'Query Time (s)':<20}"
    f"{'Top-5 Neighbor Indices'}"
)
print("-" * 100)

for name, build_t, query_t, indices in comparison:
    print(
        f"{name:<15}"
        f"{build_t:<20.4f}"
        f"{query_t:<20.4f}"
        f"{indices}"
    )

print("-" * 100)


=== TABEL PERBANDINGAN ===
----------------------------------------------------------------------------------------------------
Metode         Build Time (s)      Query Time (s)      Top-5 Neighbor Indices
----------------------------------------------------------------------------------------------------
Exact NN       0.0352              0.0869              [394553, 764272, 837727, 749223, 61511]
Annoy          9.8582              0.0003              [764272, 837727, 749223, 61511, 625227]
HNSW           138.6128            0.0005              [394553, 764272, 837727, 749223, 61511]
FAISS IVF      0.4933              0.0034              [394553, 764272, 837727, 749223, 61511]
----------------------------------------------------------------------------------------------------


**Analisis:**

Exact NN menghasilkan tetangga terdekat secara akurat, sedangkan Annoy, HNSW, dan FAISS IVF mempercepat pencarian menggunakan indeks. Perbedaan waktu komputasi dan hasil tetangga menunjukkan bahwa setiap metode memiliki karakteristik kinerja yang berbeda.